LIBS


In [ ]:
!apt update -q && apt install -y ffmpeg
!pip uninstall -y openai-whisper -q
!pip install -U "openai-whisper==20240930"

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [343 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,138 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InReleas

TRADUCIR AL ESPANOL NADA MAS


In [ ]:
import whisper
from pathlib import Path
from datetime import timedelta
import subprocess
import torch

# ================================
# TUS RUTAS
# ================================
VIDEO_PATH = "/content/drive/MyDrive/pruebavideo.mp4"
CARPETA_SALIDA = "/content/OOO"
MODELO = "medium"  # ← Perfecto para 12GB GPU

Path(CARPETA_SALIDA).mkdir(exist_ok=True)

# --- VERIFICAR Y LIMPIAR GPU ---
if torch.cuda.is_available():
    print(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()
    device = "cuda"
else:
    print("⚠️ GPU no disponible, usando CPU")
    device = "cpu"

# --- CARGAR MODELO EN GPU ---
print(f"Cargando modelo {MODELO} en {device.upper()}...")
model = whisper.load_model(MODELO, device=device)

# --- TRADUCIR A RUSO CON GPU ---
print("Traduciendo a ruso con GPU (10-30x más rápido)...")
result_ru = model.transcribe(
    VIDEO_PATH,
    language="en",
    task="translate",
    verbose=True,  # ← Cambiado a True para ver progreso
    fp16=(device == "cuda")  # ← Precisión FP16 en GPU = 2x más rápido
)

# --- GENERAR SRT EN RUSO ---
srt_path = Path(CARPETA_SALIDA) / "subtitulos_ruso.srt"

def format_time(s):
    td = timedelta(seconds=float(s))
    h, r = divmod(td.total_seconds(), 3600)
    m, sec = divmod(r, 60)
    ms = int((sec - int(sec)) * 1000)
    return f"{int(h):02}:{int(m):02}:{int(sec):02},{ms:03}"

with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(result_ru["segments"], 1):
        f.write(f"{i}\n")
        f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
        f.write(f"{seg['text'].strip()}\n\n")

print(f"✅ SRT RUSO listo: {srt_path}")

# --- GRABAR SUBTÍTULOS CON GPU (NVENC) ---
video_salida = Path(CARPETA_SALIDA) / "video_con_ruso.mp4"
print("Quemando subtítulos con aceleración GPU...")

cmd = [
    "ffmpeg", "-y",
    "-hwaccel", "cuda",  # ← Decodificación GPU
    "-i", VIDEO_PATH,
    "-vf", f"subtitles='{srt_path}':force_style='Fontsize=22,PrimaryColour=&HFFFF00&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment=2,MarginV=10'",
    "-c:v", "h264_nvenc",  # ← Codificación GPU (NVENC)
    "-preset", "p4",  # ← Preset rápido NVENC
    "-crf", "23",
    "-c:a", "copy",
    str(video_salida)
]

try:
    subprocess.run(cmd, check=True)
    print(f"✅ VIDEO listo: {video_salida}")
except subprocess.CalledProcessError:
    print("⚠️ NVENC no disponible, usando CPU...")
    cmd_cpu = [
        "ffmpeg", "-y", "-i", VIDEO_PATH,
        "-vf", f"subtitles='{srt_path}':force_style='Fontsize=22,PrimaryColour=&HFFFF00&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment=2,MarginV=10'",
        "-c:v", "libx264", "-preset", "faster", "-crf", "23",
        "-c:a", "copy", str(video_salida)
    ]
    subprocess.run(cmd_cpu, check=True)
    print(f"✅ VIDEO listo (CPU): {video_salida}")

print("🚀 ¡PROCESO COMPLETADO CON GPU!")

✅ GPU detectada: Tesla T4
   VRAM disponible: 15.8 GB
Cargando modelo medium en CUDA...
Traduciendo a ruso con GPU (10-30x más rápido)...
[00:00.000 --> 00:02.000]  Let's get out of this room.
[00:02.000 --> 00:04.000]  Oh man.
[00:04.000 --> 00:08.000]  I really hope this isn't bad news.
[00:08.000 --> 00:10.000]  I have to play it confident.
[00:10.000 --> 00:16.000]  Yes, let's play it cool. I mean our numbers weren't great, but I think if we play it right, we should be alright.
[00:16.000 --> 00:18.000]  We can talk the talk.
[00:18.000 --> 00:23.000]  Hi, Williams and Stevens, we have a one o'clock appointment with Ms. Long.
[00:23.000 --> 00:26.000]  Okay, Williams and Stevens, right?
[00:26.000 --> 00:27.000]  Yes.
[00:27.000 --> 00:30.000]  Just one second, let me make sure she's ready for you guys.
[00:40.000 --> 00:42.000]  Okay, go ahead and follow me.
[00:42.000 --> 00:44.000]  Okay, ready for us?
[00:44.000 --> 00:45.000]  Yeah.
[00:45.000 --> 00:46.000]  Excellent.
[00:50

In [ ]:
!pip install SpeechRecognition pydub googletrans==3.1.0a0

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.6 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-3.1.0a0-py3-none-any.whl size=16353 sha256=faa056a60734793ecaff640e7b97300b794b8c0d2acb2874c9ec86a93aae1fa5
  Stored in directory: /root/.cache/pip/wheels/96/ac/bd/9df9eab356c0576896e97147425987f6f45e9e46456c978d18
Successfully built googletrans
  Attempting uninstall: hyperfram

🚀 Usando: CUDA
🎤 Transcribiendo audio en inglés con Whisper...


100%|█████████████████████████████████████| 1.42G/1.42G [00:34<00:00, 43.9MiB/s]


[00:00.000 --> 00:08.000]  Hi, it's Lexi Luna. Catch me on October 30th at 3 p.m. Eastern Time on Naked News Live.
[00:08.000 --> 00:14.000]  We'll be doing a sexy little Pilates show, naked of course. See you then!
✅ SRT INGLÉS generado: /content/OOO/subtitulos_ingles.srt
🇷🇺 Traduciendo SRT a ruso con GoogleTranslate...
📋 Encontrados 2 segmentos para traducir
🔄 Traduciendo segmentos...
✅ SRT RUSO generado: /content/OOO/trans-ru.srt
✅ SRT DUAL (RU+EN) generado: /content/OOO/subtitulos_dual.srt
🎬 Generando videos con subtítulos...
📹 Generando video SOLO INGLÉS (abajo)...
✅ video SOLO INGLÉS (abajo) completado: /content/OOO/video_ingles.mp4
📹 Generando video SOLO RUSO (arriba)...
✅ video SOLO RUSO (arriba) completado: /content/OOO/video_ruso.mp4
📹 Generando video DUAL RU+EN...
✅ video DUAL RU+EN completado: /content/OOO/video_dual_ru_en.mp4

🎉 ¡PROCESO COMPLETADO!
📄 SRT Inglés:        /content/OOO/subtitulos_ingles.srt
📄 SRT Ruso:          /content/OOO/trans-ru.srt
📄 SRT Dual (RU+EN):  /

TRADUCCION DE 2 LANGUAJES

In [ ]:
import whisper
from pathlib import Path
from datetime import timedelta
import subprocess
import torch
from googletrans import Translator
import time
import re

# ================================
# TUS RUTAS
# ================================
VIDEO_PATH = "/content/Xvideos_jada_stevens_makes_a_cumback_360p.mp4"
CARPETA_SALIDA = "/content/XXX"
MODELO = "medium"

Path(CARPETA_SALIDA).mkdir(exist_ok=True)

# --- VERIFICAR GPU ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Usando: {device.upper()}")
if device == "cuda":
    torch.cuda.empty_cache()

# ================================
# PASO 1: TRANSCRIBIR CON WHISPER
# ================================
print("🎤 Transcribiendo audio en inglés con Whisper...")
model = whisper.load_model(MODELO, device=device)

result_en = model.transcribe(
    VIDEO_PATH,
    language="en",
    verbose=True,
    fp16=(device == "cuda")
)

# --- GENERAR SRT EN INGLÉS ---
srt_en_path = Path(CARPETA_SALIDA) / "subtitulos_ingles.srt"

def format_time(s):
    td = timedelta(seconds=float(s))
    h, r = divmod(td.total_seconds(), 3600)
    m, sec = divmod(r, 60)
    ms = int((sec - int(sec)) * 1000)
    return f"{int(h):02}:{int(m):02}:{int(sec):02},{ms:03}"

with open(srt_en_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(result_en["segments"], 1):
        f.write(f"{i}\n")
        f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
        f.write(f"{seg['text'].strip()}\n\n")

print(f"✅ SRT INGLÉS generado: {srt_en_path}")

# ================================
# PASO 2: LEER Y TRADUCIR SRT A RUSO
# ================================
print("🇷🇺 Traduciendo SRT a ruso con GoogleTranslate...")

def leer_srt(srt_path):
    """Lee un archivo SRT y devuelve lista de segmentos"""
    with open(srt_path, 'r', encoding='utf-8') as f:
        contenido = f.read()

    # Patrón para extraer segmentos SRT
    patron = r'(\d+)\n(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})\n(.*?)(?=\n\n|\Z)'
    segmentos = re.findall(patron, contenido, re.DOTALL)

    return [
        {
            'numero': int(seg[0]),
            'inicio': seg[1],
            'fin': seg[2],
            'texto': seg[3].strip()
        }
        for seg in segmentos
    ]

# Leer SRT en inglés
segmentos_en = leer_srt(srt_en_path)
print(f"📋 Encontrados {len(segmentos_en)} segmentos para traducir")

# Traducir con GoogleTranslate
translator = Translator()
srt_ru_path = Path(CARPETA_SALIDA) / "trans-ru.srt"
srt_dual_path = Path(CARPETA_SALIDA) / "subtitulos_dual.srt"
segmentos_ru = []

print("🔄 Traduciendo segmentos...")
for i, seg in enumerate(segmentos_en, 1):
    texto_ingles = seg['texto']

    # Traducir con reintentos
    max_intentos = 3
    for intento in range(max_intentos):
        try:
            traduccion = translator.translate(texto_ingles, src='en', dest='ru')
            texto_ruso = traduccion.text
            break
        except Exception as e:
            if intento == max_intentos - 1:
                print(f"⚠️ Error en segmento {i}: {e}")
                texto_ruso = texto_ingles  # Fallback
            else:
                time.sleep(1)

    segmentos_ru.append({
        'numero': seg['numero'],
        'inicio': seg['inicio'],
        'fin': seg['fin'],
        'texto_ru': texto_ruso,
        'texto_en': texto_ingles
    })

    # Mostrar progreso
    if i % 10 == 0:
        print(f"  ✓ {i}/{len(segmentos_en)} segmentos traducidos")

    time.sleep(0.2)

# Escribir SRT solo en ruso
with open(srt_ru_path, 'w', encoding='utf-8') as f:
    for seg in segmentos_ru:
        f.write(f"{seg['numero']}\n")
        f.write(f"{seg['inicio']} --> {seg['fin']}\n")
        f.write(f"{seg['texto_ru']}\n\n")

# Escribir SRT DUAL (Ruso + Inglés)
with open(srt_dual_path, 'w', encoding='utf-8') as f:
    for seg in segmentos_ru:
        f.write(f"{seg['numero']}\n")
        f.write(f"{seg['inicio']} --> {seg['fin']}\n")
        f.write(f"{seg['texto_ru']}\n")  # Ruso
        f.write(f"{seg['texto_en']}\n\n")  # Inglés

print(f"✅ SRT RUSO generado: {srt_ru_path}")
print(f"✅ SRT DUAL (RU+EN) generado: {srt_dual_path}")

# ================================
# PASO 3: GENERAR VIDEOS
# ================================
print("🎬 Generando videos con subtítulos...")

video_en = Path(CARPETA_SALIDA) / "video_ingles.mp4"
video_ru = Path(CARPETA_SALIDA) / "video_ruso.mp4"
video_dual = Path(CARPETA_SALIDA) / "video_dual_ru_en.mp4"

def grabar_subtitulos_dual(video_in, srt_ru, srt_en, video_out, titulo):
    """
    Graba subtítulos DUALES con posiciones diferentes
    Ruso: alignment=6 (medio derecha)
    Inglés: alignment=2 (abajo centro)
    """
    print(f"📹 Generando {titulo}...")

    # Dos filtros de subtítulos superpuestos
    style_ru = "Fontsize=22,PrimaryColour=&HFFFF00&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment=6,MarginV=0"
    style_en = "Fontsize=22,PrimaryColour=&HFFFF00&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment=2,MarginV=0"

    # Filtro compuesto: primero ruso, luego inglés
    vf_filter = f"subtitles='{srt_ru}':force_style='{style_ru}',subtitles='{srt_en}':force_style='{style_en}'"

    cmd = [
        "ffmpeg", "-y", "-hwaccel", "cuda", "-i", str(video_in),
        "-vf", vf_filter,
        "-c:v", "h264_nvenc", "-preset", "p4", "-crf", "23",
        "-c:a", "copy", str(video_out)
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError:
        print("⚠️ NVENC no disponible, usando CPU...")
        cmd_cpu = [
            "ffmpeg", "-y", "-i", str(video_in),
            "-vf", vf_filter,
            "-c:v", "libx264", "-preset", "faster", "-crf", "23",
            "-c:a", "copy", str(video_out)
        ]
        subprocess.run(cmd_cpu, check=True)

    print(f"✅ {titulo} completado: {video_out}")


# Generar videos individuales
print("\n🎬 Video 1: Solo INGLÉS (abajo centro)")
grabar_subtitulos(VIDEO_PATH, srt_en_path, video_en,
                 "video SOLO INGLÉS", alignment=2)

print("\n🎬 Video 2: Solo RUSO (medio derecha)")
grabar_subtitulos(VIDEO_PATH, srt_ru_path, video_ru,
                 "video SOLO RUSO", alignment=6)

print("\n🎬 Video 3: DUAL (RU medio derecha + EN abajo)")
grabar_subtitulos_dual(VIDEO_PATH, srt_ru_path, srt_en_path, video_dual,
                       "video DUAL RU+EN")

# ================================
# RESUMEN FINAL
# ================================
print("\n" + "="*70)
print("🎉 ¡PROCESO COMPLETADO!")
print("="*70)
print(f"📄 SRT Inglés:        {srt_en_path}")
print(f"📄 SRT Ruso:          {srt_ru_path}")
print(f"📄 SRT Dual (RU+EN):  {srt_dual_path}")
print()
print(f"🎬 Video Inglés:      {video_en}")
print(f"   └─ Alignment=2 (⬇️ ABAJO CENTRO)")
print()
print(f"🎬 Video Ruso:        {video_ru}")
print(f"   └─ Alignment=6 (➡️ MEDIO DERECHA)")
print()
print(f"🎬 Video Dual:        {video_dual}")
print(f"   └─ Ruso + Inglés")
print("="*70)
print(f"📊 Total de segmentos: {len(segmentos_en)}")
print(f"🚀 Whisper usó: {device.upper()}")

🚀 Usando: CUDA
🎤 Transcribiendo audio en inglés con Whisper...
[00:00.000 --> 00:02.000]  No?
[00:03.760 --> 00:05.760]  Holy shit.
[00:06.080 --> 00:10.080]  Galapagos, you're letting her get away, bro. Come on. Oh my god. Look at her ass though.
[00:12.240 --> 00:15.840]  I don't know. What do you tell her? What do you say to that one right there? What do you say to her?
[00:16.560 --> 00:18.560]  I don't know if we can ever catch her.
[00:18.560 --> 00:19.760]  How much?
[00:19.760 --> 00:21.760]  If we can ever. Galapagos, you got the right idea.
[00:22.800 --> 00:24.800]  We're out there. Hey, how much?
[00:24.800 --> 00:27.680]  If we can ever fucking catch her, bro. This guy's a liar.
[00:30.080 --> 00:32.080]  Oh my god, look at her ass.
[00:32.160 --> 00:34.400]  How much? How much? Hey.
[00:35.280 --> 00:40.880]  Hello. Oh my god. Oh shit. Oh shit. I just ran into Bang Bros. What? That's Jada.
[00:40.880 --> 00:44.080]  You know who it is? Oh shit, that's Jada, bro. That's wh

2 LENGUAJES + IPA

In [ ]:
!pip install SpeechRecognition pydub googletrans==3.1.0a0 epitran

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.4/222.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.3 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-3

TRAD Y COMO SUENA NO COMO SE DICE EN IPA

In [ ]:
# ================================
# 1. INSTALACIÓN
# ================================
# !pip install -q openai-whisper
# !pip install -q googletrans==4.0.0-rc1
# !pip install -q eng-to-ipa pronouncing cyrtranslit nltk
# !apt-get update > /dev/null 2>&1 && apt-get install -y ffmpeg > /dev/null 2>&1

import nltk
nltk.download('cmudict', quiet=True)

# ================================
# 2. IMPORTS
# ================================
import whisper
from pathlib import Path
from datetime import timedelta
import subprocess
import torch
from googletrans import Translator
import time
import re
import pronouncing
from eng_to_ipa import eng_to_ipa
from cyrtranslit import to_latin

# ================================
# 3. CONFIGURACIÓN
# ================================
VIDEO_PATH = "/content/Discover Lexi Luna on Naked News Live! - Tempted (1080p, h264, youtube).mp4"  # CAMBIA SI ES OTRO
CARPETA_SALIDA = "/content/OLX"
MODELO = "medium"  # tiny, base, small, medium, large

Path(CARPETA_SALIDA).mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device.upper()}")

# ================================
# 4. FUNCIONES AUXILIARES
# ================================
def format_time(s):
    td = timedelta(seconds=float(s))
    h, r = divmod(td.total_seconds(), 3600)
    m, sec = divmod(r, 60)
    ms = int((sec - int(sec)) * 1000)
    return f"{int(h):02}:{int(m):02}:{int(sec):02},{ms:03}"

def leer_srt(srt_path):
    if not srt_path.exists():
        raise FileNotFoundError(f"No existe: {srt_path}")
    with open(srt_path, 'r', encoding='utf-8') as f:
        contenido = f.read()
    patron = r'(\d+)\n(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})\n(.*?)(?=\n\n|\Z)'
    segmentos = re.findall(patron, contenido, re.DOTALL)
    return [
        {'numero': int(s[0]), 'inicio': s[1], 'fin': s[2], 'texto': s[3].strip()}
        for s in segmentos
    ]

# ================================
# 5. TRANSCRIPCIÓN WHISPER
# ================================
print("Transcribiendo con Whisper...")
model = whisper.load_model(MODELO, device=device)
result = model.transcribe(VIDEO_PATH, language="en", fp16=(device=="cuda"))

srt_en_path = Path(CARPETA_SALIDA) / "subtitulos_ingles.srt"
with open(srt_en_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(result["segments"], 1):
        f.write(f"{i}\n")
        f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
        f.write(f"{seg['text'].strip()}\n\n")
print(f"SRT inglés: {srt_en_path}")

# ================================
# 6. PRONUNCIACIÓN "ESPEJO" DEL TEXTO ORIGINAL
# ================================
segmentos_en = leer_srt(srt_en_path)
print(f"{len(segmentos_en)} segmentos")

cache_en = {}
cache_ru = {}

def ipa_to_simple(ipa):
    if not ipa: return ""
    mapeo = {
        "æ": "e", "ɑ": "a", "ʌ": "a", "ɔ": "o", "ə": "e",
        "ɪ": "i", "ʊ": "u", "eɪ": "ei", "aɪ": "ai", "ɔɪ": "oi",
        "θ": "z", "ð": "d", "ʃ": "sh", "ʒ": "zh", "tʃ": "ch", "dʒ": "j", "ŋ": "ng"
    }
    for k, v in mapeo.items():
        ipa = re.sub(k, v, ipa, flags=re.IGNORECASE)
    return re.sub(r"[ˈˌ·\s]", "", ipa.lower()).strip()

def pronunciacion_ingles(texto):
    """[ hhai, its lexi luna. kech mi an aktouber 30th... ]"""
    if not texto.strip(): return ""
    tokens = re.findall(r"\w+(?:'\w+)?|[^\w\s]", texto)
    guia = []
    for token in tokens:
        if re.match(r"[^\w']", token):
            guia.append(token)
            continue
        palabra = token.lower()
        if palabra in cache_en:
            pron = cache_en[palabra]
        else:
            try:
                ipa = eng_to_ipa(palabra)
                if ipa and "*" not in ipa:
                    pron = ipa_to_simple(ipa)
                else: pron = palabra
            except: pron = palabra
            if pron == palabra:
                try:
                    phones = pronouncing.phones_for_word(palabra)
                    if phones:
                        arp = re.sub(r'[0-9]', '', phones[0])
                        pron = arp.replace('AA','a').replace('AE','e').replace('AH','a')\
                                  .replace('AO','o').replace('AW','au').replace('AY','ai')\
                                  .replace('EH','e').replace('ER','er').replace('EY','ei')\
                                  .replace('IH','i').replace('IY','i').replace('OW','ou')\
                                  .replace('OY','oi').replace('UH','u').replace('UW','u')
                        pron = re.sub(r"[ˈˌ·\s]", "", pron.lower())
                except: pron = palabra
            cache_en[palabra] = pron
        if token[0].isupper() and len(token) > 1:
            pron = pron.capitalize()
        guia.append(pron)
    return " ".join(guia).strip()

def pronunciacion_ruso(texto):
    """[ privet, eto leksi luna. vstrechayte menya... ]"""
    if not texto.strip(): return ""
    tokens = re.findall(r"\w+|[^\w\s]", texto)
    guia = []
    for token in tokens:
        if re.match(r"[^\w]", token):
            guia.append(token)
            continue
        palabra = token
        if palabra in cache_ru:
            pron = cache_ru[palabra]
        else:
            try:
                roman = to_latin(palabra, "ru").lower()
                roman = roman.replace("kh","j").replace("zh","sh").replace("ch","ch")
                pron = re.sub(r"[ˈˌ·\s]", "", roman)
            except:
                pron = palabra.lower()
            cache_ru[palabra] = pron
        if token[0].isupper():
            pron = pron.capitalize()
        guia.append(pron)
    return " ".join(guia).strip()

# Pre-cargar
print("Pre-cargando pronunciación...")
for seg in segmentos_en:
    pronunciacion_ingles(seg['texto'])

translator = Translator()
segmentos_finales = []

print("Traduciendo y generando guías...")
for i, seg in enumerate(segmentos_en, 1):
    en = seg['texto']
    guia_en = pronunciacion_ingles(en)

    ru = en
    for _ in range(3):
        try:
            ru = translator.translate(en, src='en', dest='ru').text
            break
        except:
            time.sleep(1)

    guia_ru = pronunciacion_ruso(ru)

    segmentos_finales.append({
        'numero': seg['numero'],
        'inicio': seg['inicio'],
        'fin': seg['fin'],
        'en': en,
        'ru': ru,
        'guia_en': guia_en,
        'guia_ru': guia_ru
    })

    if i % 10 == 0:
        print(f"  {i}/{len(segmentos_en)} → {guia_en[:60]}...")

# ================================
# 7. GENERAR SRT
# ================================
def escribir_srt(ruta, lineas_func):
    with open(ruta, 'w', encoding='utf-8') as f:
        for s in segmentos_finales:
            f.write(f"{s['numero']}\n")
            f.write(f"{s['inicio']} --> {s['fin']}\n")
            for linea in lineas_func(s):
                f.write(linea + "\n")
            f.write("\n")

# Rutas
srt_eng = Path(CARPETA_SALIDA) / "sub-eng.srt"
srt_rus = Path(CARPETA_SALIDA) / "sub-rus.srt"
srt_eng_suena = Path(CARPETA_SALIDA) / "sub-eng-suena.srt"
srt_rus_suena = Path(CARPETA_SALIDA) / "sub-rus-suena.srt"

# 1. Solo inglés
escribir_srt(srt_eng, lambda s: [s['en']])

# 2. Solo ruso
escribir_srt(srt_rus, lambda s: [s['ru']])

# 3. Inglés + [guía]
escribir_srt(srt_eng_suena, lambda s: [
    s['en'],
    f"[ {s['guia_en']} ]" if s['guia_en'] else ""
])

# 4. Ruso + [guía]
escribir_srt(srt_rus_suena, lambda s: [
    s['ru'],
    f"[ {s['guia_ru']} ]" if s['guia_ru'] else ""
])

# ================================
# 8. GENERAR 5 VIDEOS (GUÍA PEQUEÑA, SIN PUNTOS)
# ================================
print("\nGenerando 5 videos...")

def grabar_simple(srt_in, video_out, titulo, align, color_text):
    print(f"   {titulo}")
    style = (
        f"Fontsize=22,PrimaryColour={color_text},OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment={align},MarginV=40,"
        f"Fontsize=16,PrimaryColour=&HAAAAAA&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment={align},MarginV=15"
    )
    cmd = [
        "ffmpeg", "-y", "-i", str(VIDEO_PATH),
        "-vf", f"subtitles='{srt_in}':force_style='{style}'",
        "-c:v", "libx264", "-crf", "23", "-preset", "faster", "-c:a", "copy", str(video_out)
    ]
    subprocess.run(cmd, check=True, capture_output=True)

# Videos simples
grabar_simple(srt_eng, Path(CARPETA_SALIDA)/"sub-eng.mp4", "sub-eng.mp4", align=2, color_text="&HFFFFFF&")
grabar_simple(srt_rus, Path(CARPETA_SALIDA)/"sub-rus.mp4", "sub-rus.mp4", align=8, color_text="&H00FFFF&")
grabar_simple(srt_eng_suena, Path(CARPETA_SALIDA)/"sub-eng-suena.mp4", "sub-eng-suena.mp4", align=2, color_text="&HFFFFFF&")
grabar_simple(srt_rus_suena, Path(CARPETA_SALIDA)/"sub-rus-suena.mp4", "sub-rus-suena.mp4", align=8, color_text="&H00FFFF&")

# --- VIDEO DUAL ---
srt_temp_ru = Path(CARPETA_SALIDA) / "temp_ru.srt"
srt_temp_en = Path(CARPETA_SALIDA) / "temp_en.srt"

escribir_srt(srt_temp_ru, lambda s: [
    s['ru'],
    f"[ {s['guia_ru']} ]" if s['guia_ru'] else ""
])

escribir_srt(srt_temp_en, lambda s: [
    s['en'],
    f"[ {s['guia_en']} ]" if s['guia_en'] else ""
])

print("   sub-eng+rus-suena.mp4")
vf = (
    f"subtitles='{srt_temp_ru}':force_style='Fontsize=22,PrimaryColour=&H00FFFF&,Alignment=8,MarginV=45',"
    f"subtitles='{srt_temp_ru}':force_style='Fontsize=16,PrimaryColour=&HAAAAAA&,Alignment=8,MarginV=20',"
    f"subtitles='{srt_temp_en}':force_style='Fontsize=22,PrimaryColour=&HFFFF00&,Alignment=2,MarginV=45',"
    f"subtitles='{srt_temp_en}':force_style='Fontsize=16,PrimaryColour=&HAAAAAA&,Alignment=2,MarginV=20'"
)
cmd = [
    "ffmpeg", "-y", "-i", str(VIDEO_PATH),
    "-vf", vf,
    "-c:v", "libx264", "-crf", "23", "-preset", "faster",
    "-c:a", "copy", str(Path(CARPETA_SALIDA)/"sub-eng+rus-suena.mp4")
]
subprocess.run(cmd, check=True, capture_output=True)

srt_temp_ru.unlink(missing_ok=True)
srt_temp_en.unlink(missing_ok=True)

# ================================
# 9. RESUMEN FINAL
# ================================
print("\n" + "="*70)
print("5 VIDEOS GENERADOS CON ÉXITO")
print("="*70)
for v in Path(CARPETA_SALIDA).glob("sub-*.mp4"):
    print(f"   {v.name}")
print(f"\nCarpeta: {CARPETA_SALIDA}")
print(f"Total segmentos: {len(segmentos_finales)}")
print("Formato guía: [ hhai, its lexi luna. kech mi an aktouber 30th... ]")
print("="*70)

Usando: CUDA
Transcribiendo con Whisper...
SRT inglés: /content/OLX/subtitulos_ingles.srt
2 segmentos
Pre-cargando pronunciación...
Traduciendo y generando guías...

Generando 5 videos...
   sub-eng.mp4
   sub-rus.mp4
   sub-eng-suena.mp4
   sub-rus-suena.mp4
   sub-eng+rus-suena.mp4

5 VIDEOS GENERADOS CON ÉXITO
   sub-rus-suena.mp4
   sub-rus.mp4
   sub-eng+rus-suena.mp4
   sub-eng-suena.mp4
   sub-eng.mp4

Carpeta: /content/OLX
Total segmentos: 2
Formato guía: [ hhai, its lexi luna. kech mi an aktouber 30th... ]


IPA  + TRAD + RUS DUAL TRANSLATED

In [ ]:
# ================================
# 1. INSTALACIÓN
# ================================
!pip install -q openai-whisper
!pip install -q googletrans==4.0.0-rc1
!pip install -q eng-to-ipa pronouncing cyrtranslit nltk
!apt-get update > /dev/null 2>&1 && apt-get install -y ffmpeg > /dev/null 2>&1

import nltk
nltk.download('cmudict', quiet=True)

# ================================
# 2. IMPORTS
# ================================
import whisper
from pathlib import Path
from datetime import timedelta
import subprocess
import torch
from googletrans import Translator
import time
import re
import pronouncing
from eng_to_ipa import eng_to_ipa
from cyrtranslit import to_latin

# ================================
# 3. CONFIGURACIÓN
# ================================
VIDEO_PATH = "/content/Discover Lexi Luna on Naked News Live! - Tempted (1080p, h264, youtube).mp4"  # ← CAMBIA POR TU VIDEO
CARPETA_SALIDA = "/content/OLXXX"
MODELO = "medium"

Path(CARPETA_SALIDA).mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device.upper()}")

# ================================
# 4. FUNCIONES AUXILIARES
# ================================
def format_time(s):
    td = timedelta(seconds=float(s))
    h, r = divmod(td.total_seconds(), 3600)
    m, sec = divmod(r, 60)
    ms = int((sec - int(sec)) * 1000)
    return f"{int(h):02}:{int(m):02}:{int(sec):02},{ms:03}"

def leer_srt(srt_path):
    if not srt_path.exists():
        raise FileNotFoundError(f"No existe: {srt_path}")
    with open(srt_path, 'r', encoding='utf-8') as f:
        contenido = f.read()
    patron = r'(\d+)\n(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})\n(.*?)(?=\n\n|\Z)'
    segmentos = re.findall(patron, contenido, re.DOTALL)
    return [
        {'numero': int(s[0]), 'inicio': s[1], 'fin': s[2], 'texto': s[3].strip()}
        for s in segmentos
    ]

# ================================
# 5. FONÉTICA REAL COMO GOOGLE (CORREGIDA)
# ================================
cache_en = {}
cache_ru = {}

def fonetica_google_ingles(palabra):
    palabra = palabra.lower().strip()
    if not palabra:
        return ""
    if palabra in cache_en:
        return cache_en[palabra]

    pron = palabra

    # 1. eng-to-ipa
    try:
        ipa = eng_to_ipa(palabra)
        if ipa and "*" not in ipa:
            pron = ipa.replace("ˈ", "").replace("ˌ", "").replace("ː", "")
            pron = re.sub(r"[·\s]", "", pron)
            mapeo = {
                "ʃ": "sh", "ʒ": "zh", "tʃ": "ch", "dʒ": "j", "θ": "th", "ð": "th",
                "ŋ": "ng", "æ": "a", "ɑ": "a", "ʌ": "u", "ɔ": "o", "ə": "u",
                "ɪ": "i", "ʊ": "u", "eɪ": "ay", "aɪ": "ai", "ɔɪ": "oy",
                "oʊ": "oh", "aʊ": "ow"
            }
            for k, v in mapeo.items():
                pron = pron.replace(k, v)
            pron = pron.lower()
    except: pass

    # 2. pronouncing fallback
    if pron == palabra:
        try:
            phones = pronouncing.phones_for_word(palabra)
            if phones:
                arp = re.sub(r'[0-9]', '', phones[0])
                pron = arp.replace(' ', '').lower()
                pron = pron.replace('aa', 'a').replace('ae', 'a').replace('ah', 'u')
                pron = pron.replace('ao', 'o').replace('aw', 'ow').replace('ay', 'ai')
                pron = pron.replace('eh', 'e').replace('er', 'er').replace('ey', 'ay')
                pron = pron.replace('ih', 'i').replace('iy', 'ee').replace('ow', 'oh')
                pron = pron.replace('oy', 'oy').replace('uh', 'u').replace('uw', 'oo')
        except: pass

    cache_en[palabra] = pron
    return pron

def fonetica_google_ruso(palabra):
    if palabra in cache_ru:
        return cache_ru[palabra]
    try:
        roman = to_latin(palabra, "ru").lower()
        roman = roman.replace("kh", "kh").replace("zh", "zh").replace("ch", "ch")
        roman = re.sub(r"[ˈˌ·\s]", "", roman)
        cache_ru[palabra] = roman
        return roman
    except:
        cache_ru[palabra] = palabra.lower()
        return palabra.lower()

def guia_fonetica(texto, idioma="en"):
    if not texto.strip():
        return ""
    tokens = re.findall(r"\w+(?:'\w+)?|[^\w\s]", texto)
    guia = []
    for token in tokens:
        if re.match(r"[^\w']", token):
            guia.append(token)
            continue
        original = token
        palabra = token.lower()
        if idioma == "en":
            pron = fonetica_google_ingles(palabra)
        else:
            pron = fonetica_google_ruso(token)
        if original[0].isupper() and len(original) > 1:
            pron = pron.capitalize()
        guia.append(pron)
    return " ".join(guia).strip()

# ================================
# 6. TRANSCRIPCIÓN + TRADUCCIÓN
# ================================
print("Transcribiendo con Whisper...")
model = whisper.load_model(MODELO, device=device)
result = model.transcribe(VIDEO_PATH, language="en", fp16=(device=="cuda"))

srt_en_path = Path(CARPETA_SALIDA) / "subtitulos_ingles.srt"
with open(srt_en_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(result["segments"], 1):
        f.write(f"{i}\n")
        f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
        f.write(f"{seg['text'].strip()}\n\n")
print(f"SRT inglés: {srt_en_path}")

segmentos_en = leer_srt(srt_en_path)
print(f"{len(segmentos_en)} segmentos")

translator = Translator()
segmentos_finales = []

print("Generando fonética real y traducción...")
for i, seg in enumerate(segmentos_en, 1):
    en = seg['texto']
    guia_en = guia_fonetica(en, "en")

    ru = en
    for _ in range(3):
        try:
            ru = translator.translate(en, src='en', dest='ru').text
            break
        except:
            time.sleep(1)

    guia_ru = guia_fonetica(ru, "ru")

    segmentos_finales.append({
        'numero': seg['numero'],
        'inicio': seg['inicio'],
        'fin': seg['fin'],
        'en': en,
        'ru': ru,
        'guia_en': guia_en,
        'guia_ru': guia_ru
    })

    if i % 10 == 0:
        print(f"  {i}/{len(segmentos_en)} → {guia_en[:60]}...")

# ================================
# 7. GENERAR SRT
# ================================
def escribir_srt(ruta, lineas_func):
    with open(ruta, 'w', encoding='utf-8') as f:
        for s in segmentos_finales:
            f.write(f"{s['numero']}\n")
            f.write(f"{s['inicio']} --> {s['fin']}\n")
            for linea in lineas_func(s):
                f.write(linea + "\n")
            f.write("\n")

srt_eng = Path(CARPETA_SALIDA) / "sub-eng.srt"
srt_rus = Path(CARPETA_SALIDA) / "sub-rus.srt"
srt_eng_suena = Path(CARPETA_SALIDA) / "sub-eng-suena.srt"
srt_rus_suena = Path(CARPETA_SALIDA) / "sub-rus-suena.srt"

escribir_srt(srt_eng, lambda s: [s['en']])
escribir_srt(srt_rus, lambda s: [s['ru']])
escribir_srt(srt_eng_suena, lambda s: [s['en'], f"[ {s['guia_en']} ]" if s['guia_en'] else ""])
escribir_srt(srt_rus_suena, lambda s: [s['ru'], f"[ {s['guia_ru']} ]" if s['guia_ru'] else ""])

# ================================
# 8. GENERAR 5 VIDEOS
# ================================
print("\nGenerando 5 videos...")

def grabar_simple(srt_in, video_out, titulo, align, color_text):
    print(f"   {titulo}")
    style = (
        f"Fontsize=22,PrimaryColour={color_text},OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment={align},MarginV=40,"
        f"Fontsize=13,PrimaryColour=&HAAAAAA&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment={align},MarginV=15"
    )
    cmd = [
        "ffmpeg", "-y", "-i", str(VIDEO_PATH),
        "-vf", f"subtitles='{srt_in}':force_style='{style}'",
        "-c:v", "libx264", "-crf", "23", "-preset", "faster", "-c:a", "copy", str(video_out)
    ]
    subprocess.run(cmd, check=True, capture_output=True)

grabar_simple(srt_eng, Path(CARPETA_SALIDA)/"sub-eng.mp4", "sub-eng.mp4", align=2, color_text="&HFFFFFF&")
grabar_simple(srt_rus, Path(CARPETA_SALIDA)/"sub-rus.mp4", "sub-rus.mp4", align=6, color_text="&H00FFFF&")
grabar_simple(srt_eng_suena, Path(CARPETA_SALIDA)/"sub-eng-suena.mp4", "sub-eng-suena.mp4", align=2, color_text="&HFFFFFF&")
grabar_simple(srt_rus_suena, Path(CARPETA_SALIDA)/"sub-rus-suena.mp4", "sub-rus-suena.mp4", align=6, color_text="&H00FFFF&")

# VIDEO DUAL
srt_temp_ru = Path(CARPETA_SALIDA) / "temp_ru.srt"
srt_temp_en = Path(CARPETA_SALIDA) / "temp_en.srt"

escribir_srt(srt_temp_ru, lambda s: [s['ru'], f"[ {s['guia_ru']} ]" if s['guia_ru'] else ""])
escribir_srt(srt_temp_en, lambda s: [s['en'], f"[ {s['guia_en']} ]" if s['guia_en'] else ""])

print("   sub-eng+rus-suena.mp4")
vf = (
    f"subtitles='{srt_temp_ru}':force_style='Fontsize=22,PrimaryColour=&H00FFFF&,Alignment=8,MarginV=45',"
    f"subtitles='{srt_temp_ru}':force_style='Fontsize=16,PrimaryColour=&HAAAAAA&,Alignment=8,MarginV=20',"
    f"subtitles='{srt_temp_en}':force_style='Fontsize=22,PrimaryColour=&HFFFF00&,Alignment=2,MarginV=45',"
    f"subtitles='{srt_temp_en}':force_style='Fontsize=16,PrimaryColour=&HAAAAAA&,Alignment=2,MarginV=20'"
)
cmd = ["ffmpeg", "-y", "-i", str(VIDEO_PATH), "-vf", vf, "-c:v", "libx264", "-crf", "23", "-c:a", "copy", str(Path(CARPETA_SALIDA)/"sub-eng+rus-suena.mp4")]
subprocess.run(cmd, check=True, capture_output=True)

srt_temp_ru.unlink(missing_ok=True)
srt_temp_en.unlink(missing_ok=True)

# ================================
# 9. RESUMEN + ZIP AUTOMÁTICO
# ================================
print("\n" + "="*70)
print("5 VIDEOS GENERADOS CON FONÉTICA REAL")
print("="*70)
for v in Path(CARPETA_SALIDA).glob("sub-*.mp4"):
    print(f"   {v.name}")

# ZIP
!zip -q -r /content/subtitulos_foneticos.zip /content/OOO
print("\nZIP generado: /content/subtitulos_foneticos.zip")
print("Descárgalo desde la barra lateral")
print("="*70)

Usando: CUDA
Transcribiendo con Whisper...
SRT inglés: /content/OLXXX/subtitulos_ingles.srt
2 segmentos
Generando fonética real y traducción...

Generando 5 videos...
   sub-eng.mp4
   sub-rus.mp4
   sub-eng-suena.mp4
   sub-rus-suena.mp4
   sub-eng+rus-suena.mp4

5 VIDEOS GENERADOS CON FONÉTICA REAL
   sub-rus-suena.mp4
   sub-rus.mp4
   sub-eng+rus-suena.mp4
   sub-eng-suena.mp4
   sub-eng.mp4

ZIP generado: /content/subtitulos_foneticos.zip
Descárgalo desde la barra lateral


In [ ]:
# ================================
# 1. INSTALACIÓN
# ================================
# !pip install -q openai-whisper
# !pip install -q googletrans==4.0.0-rc1
# !pip install -q eng-to-ipa pronouncing cyrtranslit nltk
# !apt-get update > /dev/null 2>&1 && apt-get install -y ffmpeg > /dev/null 2>&1

import nltk
nltk.download('cmudict', quiet=True)

# ================================
# 2. IMPORTS
# ================================
import whisper
from pathlib import Path
from datetime import timedelta
import subprocess
import torch
from googletrans import Translator
import time
import re
import pronouncing
from eng_to_ipa import eng_to_ipa
from cyrtranslit import to_latin

# ================================
# 3. CONFIGURACIÓN
# ================================
VIDEO_PATH = "/content/Discover Lexi Luna on Naked News Live! - Tempted (1080p, h264, youtube).mp4"  # ← CAMBIA POR TU VIDEO
CARPETA_SALIDA = "/content/OLXXXX"
MODELO = "medium"

Path(CARPETA_SALIDA).mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device.upper()}")

# ================================
# 4. FUNCIONES AUXILIARES
# ================================
def format_time(s):
    td = timedelta(seconds=float(s))
    h, r = divmod(td.total_seconds(), 3600)
    m, sec = divmod(r, 60)
    ms = int((sec - int(sec)) * 1000)
    return f"{int(h):02}:{int(m):02}:{int(sec):02},{ms:03}"

def leer_srt(srt_path):
    if not srt_path.exists():
        raise FileNotFoundError(f"No existe: {srt_path}")
    with open(srt_path, 'r', encoding='utf-8') as f:
        contenido = f.read()
    patron = r'(\d+)\n(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})\n(.*?)(?=\n\n|\Z)'
    segmentos = re.findall(patron, contenido, re.DOTALL)
    return [
        {'numero': int(s[0]), 'inicio': s[1], 'fin': s[2], 'texto': s[3].strip()}
        for s in segmentos
    ]

# ================================
# 5. FONÉTICA REAL COMO GOOGLE
# ================================
cache_en = {}
cache_ru = {}

def fonetica_google_ingles(palabra):
    palabra = palabra.lower().strip()
    if not palabra:
        return ""
    if palabra in cache_en:
        return cache_en[palabra]

    pron = palabra

    try:
        ipa = eng_to_ipa(palabra)
        if ipa and "*" not in ipa:
            pron = ipa.replace("ˈ", "").replace("ˌ", "").replace("ː", "")
            pron = re.sub(r"[·\s]", "", pron)
            mapeo = {
                "ʃ": "sh", "ʒ": "zh", "tʃ": "ch", "dʒ": "j", "θ": "th", "ð": "th",
                "ŋ": "ng", "æ": "a", "ɑ": "a", "ʌ": "u", "ɔ": "o", "ə": "u",
                "ɪ": "i", "ʊ": "u", "eɪ": "ay", "aɪ": "ai", "ɔɪ": "oy",
                "oʊ": "oh", "aʊ": "ow"
            }
            for k, v in mapeo.items():
                pron = pron.replace(k, v)
            pron = pron.lower()
    except: pass

    if pron == palabra:
        try:
            phones = pronouncing.phones_for_word(palabra)
            if phones:
                arp = re.sub(r'[0-9]', '', phones[0])
                pron = arp.replace(' ', '').lower()
                pron = pron.replace('aa', 'a').replace('ae', 'a').replace('ah', 'u')
                pron = pron.replace('ao', 'o').replace('aw', 'ow').replace('ay', 'ai')
                pron = pron.replace('eh', 'e').replace('er', 'er').replace('ey', 'ay')
                pron = pron.replace('ih', 'i').replace('iy', 'ee').replace('ow', 'oh')
                pron = pron.replace('oy', 'oy').replace('uh', 'u').replace('uw', 'oo')
        except: pass

    cache_en[palabra] = pron
    return pron

def fonetica_google_ruso(palabra):
    if palabra in cache_ru:
        return cache_ru[palabra]
    try:
        roman = to_latin(palabra, "ru").lower()
        roman = roman.replace("kh", "kh").replace("zh", "zh").replace("ch", "ch")
        roman = re.sub(r"[ˈˌ·\s]", "", roman)
        cache_ru[palabra] = roman
        return roman
    except:
        cache_ru[palabra] = palabra.lower()
        return palabra.lower()

def guia_fonetica(texto, idioma="en"):
    if not texto.strip():
        return ""
    tokens = re.findall(r"\w+(?:'\w+)?|[^\w\s]", texto)
    guia = []
    for token in tokens:
        if re.match(r"[^\w']", token):
            guia.append(token)
            continue
        original = token
        palabra = token.lower()
        if idioma == "en":
            pron = fonetica_google_ingles(palabra)
        else:
            pron = fonetica_google_ruso(token)
        if original[0].isupper() and len(original) > 1:
            pron = pron.capitalize()
        guia.append(pron)
    return " ".join(guia).strip()

# ================================
# 6. TRANSCRIPCIÓN + TRADUCCIÓN
# ================================
print("Transcribiendo con Whisper...")
model = whisper.load_model(MODELO, device=device)
result = model.transcribe(VIDEO_PATH, language="en", fp16=(device=="cuda"))

srt_en_path = Path(CARPETA_SALIDA) / "subtitulos_ingles.srt"
with open(srt_en_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(result["segments"], 1):
        f.write(f"{i}\n")
        f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
        f.write(f"{seg['text'].strip()}\n\n")
print(f"SRT inglés: {srt_en_path}")

segmentos_en = leer_srt(srt_en_path)
print(f"{len(segmentos_en)} segmentos")

translator = Translator()
segmentos_finales = []

print("Generando fonética real y traducción...")
for i, seg in enumerate(segmentos_en, 1):
    en = seg['texto']
    guia_en = guia_fonetica(en, "en")

    ru = en
    for _ in range(3):
        try:
            ru = translator.translate(en, src='en', dest='ru').text
            break
        except:
            time.sleep(1)

    guia_ru = guia_fonetica(ru, "ru")

    segmentos_finales.append({
        'numero': seg['numero'],
        'inicio': seg['inicio'],
        'fin': seg['fin'],
        'en': en,
        'ru': ru,
        'guia_en': guia_en,
        'guia_ru': guia_ru
    })

    if i % 10 == 0:
        print(f"  {i}/{len(segmentos_en)} → {guia_en[:60]}...")

# ================================
# 7. GENERAR SRT
# ================================
def escribir_srt(ruta, lineas_func):
    with open(ruta, 'w', encoding='utf-8') as f:
        for s in segmentos_finales:
            f.write(f"{s['numero']}\n")
            f.write(f"{s['inicio']} --> {s['fin']}\n")
            for linea in lineas_func(s):
                f.write(linea + "\n")
            f.write("\n")

srt_eng = Path(CARPETA_SALIDA) / "sub-eng.srt"
srt_rus = Path(CARPETA_SALIDA) / "sub-rus.srt"
srt_eng_suena = Path(CARPETA_SALIDA) / "sub-eng-suena.srt"
srt_rus_suena = Path(CARPETA_SALIDA) / "sub-rus-suena.srt"

escribir_srt(srt_eng, lambda s: [s['en']])
escribir_srt(srt_rus, lambda s: [s['ru']])
escribir_srt(srt_eng_suena, lambda s: [s['en'], f"[ {s['guia_en']} ]" if s['guia_en'] else ""])
escribir_srt(srt_rus_suena, lambda s: [s['ru'], f"[ {s['guia_ru']} ]" if s['guia_ru'] else ""])

# ================================
# 8. SELECCIÓN DE VIDEOS CON CHECKBOX
# ================================
print("\n" + "="*60)
print("SELECCIONA LOS VIDEOS QUE QUIERES GENERAR")
print("="*60)

videos = {
    "sub-eng.mp4": {
        "srt": srt_eng,
        "out": Path(CARPETA_SALIDA)/"sub-eng.mp4",
        "align": 2,
        "color": "&HFFFFFF&"
    },
    "sub-rus.mp4": {
        "srt": srt_rus,
        "out": Path(CARPETA_SALIDA)/"sub-rus.mp4",
        "align": 6,
        "color": "&H00FFFF&"
    },
    "sub-eng-suena.mp4": {
        "srt": srt_eng_suena,
        "out": Path(CARPETA_SALIDA)/"sub-eng-suena.mp4",
        "align": 2,
        "color": "&HFFFFFF&"
    },
    "sub-rus-suena.mp4": {
        "srt": srt_rus_suena,
        "out": Path(CARPETA_SALIDA)/"sub-rus-suena.mp4",
        "align": 6,
        "color": "&H00FFFF&"
    },
    "sub-eng+rus-suena.mp4": {
        "dual": True,
        "out": Path(CARPETA_SALIDA)/"sub-eng+rus-suena.mp4"
    }
}

# Mostrar menú
print("Escribe 'x' para seleccionar, Enter para dejar vacío:")
seleccion = {}
for nombre in videos:
    marca = input(f"  {nombre} () ").strip().lower()
    seleccion[nombre] = marca == "x"

# ================================
# 9. GENERAR SOLO LOS SELECCIONADOS
# ================================
def grabar_simple(srt_in, video_out, align, color_text):
    print(f"   Generando: {video_out.name}")
    style = (
        f"Fontsize=22,PrimaryColour={color_text},OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment={align},MarginV=40,"
        f"Fontsize=13,PrimaryColour=&HAAAAAA&,OutlineColour=&H000000&,BackColour=&H80000000&,BorderStyle=1,Alignment={align},MarginV=15"
    )
    cmd = [
        "ffmpeg", "-y", "-i", str(VIDEO_PATH),
        "-vf", f"subtitles='{srt_in}':force_style='{style}'",
        "-c:v", "libx264", "-crf", "23", "-preset", "faster", "-c:a", "copy", str(video_out)
    ]
    subprocess.run(cmd, check=True, capture_output=True)

generados = []

# Videos simples
if seleccion["sub-eng.mp4"]:
    grabar_simple(videos["sub-eng.mp4"]["srt"], videos["sub-eng.mp4"]["out"], 2, "&HFFFFFF&")
    generados.append("sub-eng.mp4")

if seleccion["sub-rus.mp4"]:
    grabar_simple(videos["sub-rus.mp4"]["srt"], videos["sub-rus.mp4"]["out"], 2, "&H00FFFF&")
    generados.append("sub-rus.mp4")

if seleccion["sub-eng-suena.mp4"]:
    grabar_simple(videos["sub-eng-suena.mp4"]["srt"], videos["sub-eng-suena.mp4"]["out"], 2, "&HFFFFFF&")
    generados.append("sub-eng-suena.mp4")

if seleccion["sub-rus-suena.mp4"]:
    grabar_simple(videos["sub-rus-suena.mp4"]["srt"], videos["sub-rus-suena.mp4"]["out"], 2, "&H00FFFF&")
    generados.append("sub-rus-suena.mp4")

# VIDEO DUAL - CORREGIDO (sin duplicados)
# ================================
if seleccion["sub-eng+rus-suena.mp4"]:
    print("   Generando: sub-eng+rus-suena.mp4")

    # Archivos separados para texto principal y guía
    srt_ru_texto = Path(CARPETA_SALIDA) / "temp_ru_texto.srt"
    srt_ru_guia = Path(CARPETA_SALIDA) / "temp_ru_guia.srt"
    srt_en_texto = Path(CARPETA_SALIDA) / "temp_en_texto.srt"
    srt_en_guia = Path(CARPETA_SALIDA) / "temp_en_guia.srt"

    # Escribir archivos SRT separados
    escribir_srt(srt_ru_texto, lambda s: [s['ru']])  # Solo texto ruso
    escribir_srt(srt_ru_guia, lambda s: [f"[ {s['guia_ru']} ]"] if s['guia_ru'] else [""])  # Solo guía rusa
    escribir_srt(srt_en_texto, lambda s: [s['en']])  # Solo texto inglés
    escribir_srt(srt_en_guia, lambda s: [f"[ {s['guia_en']} ]"] if s['guia_en'] else [""])  # Solo guía inglesa

    vf = (
      # RUSO - Texto principal (celeste, arriba)
      f"subtitles='{srt_ru_texto}':force_style='Fontsize=18,PrimaryColour=&H00FFFF&,Bold=1,OutlineColour=&H000000&,Outline=2,BorderStyle=1,Alignment=6,MarginV=0',"
      # RUSO - Guía fonética (gris claro, más abajo)
      f"subtitles='{srt_ru_guia}':force_style='Fontsize=16,PrimaryColour=&HAAAAAA&,Italic=1,OutlineColour=&H000000&,Outline=2,BorderStyle=1,Alignment=6,MarginV=30',"
      # INGLÉS - Texto principal (amarillo, abajo)
      f"subtitles='{srt_en_texto}':force_style='Fontsize=18,PrimaryColour=&HFFFF00&,Bold=1,OutlineColour=&H000000&,Outline=2,BorderStyle=1,Alignment=2,MarginV=30',"
      # INGLÉS - Guía fonética (gris claro, más arriba del texto inglés)
      f"subtitles='{srt_en_guia}':force_style='Fontsize=16,PrimaryColour=&HAAAAAA&,Italic=1,OutlineColour=&H000000&,Outline=2,BorderStyle=1,Alignment=2,MarginV=0'"
      )

    cmd = [
        "ffmpeg", "-y", "-i", str(VIDEO_PATH),
        "-vf", vf,
        "-c:v", "libx264", "-crf", "23", "-preset", "faster",
        "-c:a", "copy", str(videos["sub-eng+rus-suena.mp4"]["out"])
    ]
    subprocess.run(cmd, check=True, capture_output=True)

    # Limpiar archivos temporales
    srt_ru_texto.unlink(missing_ok=True)
    srt_ru_guia.unlink(missing_ok=True)
    srt_en_texto.unlink(missing_ok=True)
    srt_en_guia.unlink(missing_ok=True)

    generados.append("sub-eng+rus-suena.mp4")

# ================================
# 10. RESUMEN + ZIP
# ================================
print("\n" + "="*70)
if generados:
    print(f"Videos generados: {len(generados)}")
    for v in generados:
        print(f"   {v}")
else:
    print("No se generó ningún video.")
print("="*70)

# ZIP solo con los generados
if generados:
    zip_path = "/content/videos_seleccionados.zip"
    !zip -q -r "{zip_path}" "{CARPETA_SALIDA}" -i "*.mp4"
    print(f"\nZIP generado: {zip_path}")
    print("Descárgalo desde la barra lateral")
else:
    print("\nNo hay videos para comprimir.")
print("="*70)

# recursos
# https://claude.ai/share/b48e4b93-52e6-4689-aa49-4a0b5736f7d7
# https://claude.ai/share/b48e4b93-52e6-4689-aa49-4a0b5736f7d7
# https://grok.com/share/bGVnYWN5LWNvcHk_19a9a416-1f36-461e-bb3c-5cf6d99ebb88

Usando: CPU
Transcribiendo con Whisper...
SRT inglés: /content/OLXXXX/subtitulos_ingles.srt
2 segmentos
Generando fonética real y traducción...

SELECCIONA LOS VIDEOS QUE QUIERES GENERAR
Escribe 'x' para seleccionar, Enter para dejar vacío:
  sub-eng.mp4 () 
  sub-rus.mp4 () 
  sub-eng-suena.mp4 () 
  sub-rus-suena.mp4 () 
  sub-eng+rus-suena.mp4 () x
   Generando: sub-eng+rus-suena.mp4

Videos generados: 1
   sub-eng+rus-suena.mp4

ZIP generado: /content/videos_seleccionados.zip
Descárgalo desde la barra lateral
